In [7]:
from qiskit import QuantumCircuit
from qiskit.transpiler import PassManager, Layout
from qiskit.transpiler.passes import InverseCancellation, CommutativeCancellation
from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping
from qiskit.circuit.library import CXGate, PauliEvolutionGate

from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, FindCommutingPauliEvolutionsMulti

from qiskit_qaoa.utils.commuting_gate_router_precompute_rzz_mask import CommutingGateRouterPrecomputeRzz as MaskRouter
from qiskit_qaoa.utils.commuting_gate_router_precompute_rzz import CommutingGateRouterPrecomputeRzz as Router
from qiskit_qaoa.hubo.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian
from qiskit_qaoa.utils.gfa_utils import gfa_file_to_graph

In [8]:
filename = 'trivial'
copy_numbers = [1,1,1]
filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{filename}.gfa'
graph, n, V, T = gfa_file_to_graph(filepath, copy_numbers)
hamiltonian = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)
ess = ExtendedSwapStrategy.from_grid(n, T)
num_physical_qubits = n * T

Keeping constraints at times: [1 0]


In [11]:
pm_mask = PassManager(
    [
        FindCommutingPauliEvolutionsMulti(), 
        MaskRouter(
            ess,
            max_layers=10,
            perform_extra_swaps=True
        ),
        SwapToFinalMapping(),
        InverseCancellation(gates_to_cancel=[CXGate()]),
        CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
        InverseCancellation(gates_to_cancel=[CXGate()]),
    ]
)
pm_no_mask = PassManager(
    [
        FindCommutingPauliEvolutionsMulti(), 
        Router(
            ess,
            max_layers=10,
            perform_extra_swaps=True
        ),
        SwapToFinalMapping(),
        InverseCancellation(gates_to_cancel=[CXGate()]),
        CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
        InverseCancellation(gates_to_cancel=[CXGate()]),
    ]
)

In [9]:
donor_qc = QuantumCircuit(num_physical_qubits)
layout = Layout({donor_qc.qubits[key]: key for key in range(num_physical_qubits)})

In [10]:
qc = QuantumCircuit(num_physical_qubits)
qc.append(PauliEvolutionGate(hamiltonian), [layout.get_virtual_bits()[donor_qc.qubits[i]] for i in range(num_physical_qubits)])


In [12]:
tqc_mask = pm_mask.run(qc)  

Max layers needed to apply swap decompose: 6
Gates we cannot directly implement: 0
Transpiling un-implemented gates


In [13]:
tqc_no_mask = pm_no_mask.run(qc)  

Max layers needed to apply swap decompose: 6
Gates we cannot directly implement: 0
Transpiling un-implemented gates


In [15]:
tqc_mask.count_ops()

OrderedDict([('cx', 120), ('rzz', 67), ('rz', 24), ('swap', 16)])

In [16]:
tqc_no_mask.count_ops()

OrderedDict([('cx', 108), ('rzz', 59), ('rz', 32), ('swap', 16)])